In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.preprocessing import (StandardScaler, LabelEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer 
from sklearn.model_selection import (train_test_split, GridSearchCV)
from sklearn.linear_model import LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.naive_bayes import GaussianNB 
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report)
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('loan_approved_cleaned.csv')
df.head()

,Applicant_Income,Coapplicant_Income,Age,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,...,Loan_Purpose_Education,Loan_Purpose_Home,Loan_Purpose_Personal,Property_Area_Semiurban,Property_Area_Urban,Gender_Male,Employer_Category_Government,Employer_Category_MNC,Employer_Category_Private,Employer_Category_Unemployed
0,17795.0,1387.0,51.0,0.0,637.0,4.0,0.5,19403.0,45638.0,16619.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,2860.0,2679.0,46.0,3.0,621.0,2.0,0.3,2580.0,49272.0,38687.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,7390.0,2106.0,25.0,2.0,674.0,4.0,0.2,13844.0,6908.0,27943.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,13964.0,8173.0,40.0,2.0,579.0,3.0,0.3,9553.0,10844.0,27819.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,13284.0,4223.0,31.0,2.0,721.0,1.0,0.3,9386.0,37629.0,12741.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


### We will train 3 models without and with feature engineering and check which model predicts best 
Logistic Regression  
KNNClassifier  
GaussianNB 


In [3]:
# Train-test split 

X = df.drop(columns=['Loan_Approved'], axis=True)
y = df['Loan_Approved'] 

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
X_test.head()

,Applicant_Income,Coapplicant_Income,Age,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,...,Loan_Purpose_Education,Loan_Purpose_Home,Loan_Purpose_Personal,Property_Area_Semiurban,Property_Area_Urban,Gender_Male,Employer_Category_Government,Employer_Category_MNC,Employer_Category_Private,Employer_Category_Unemployed
29,5890.0,8041.0,31.0,0.0,603.0,0.0,0.1,11906.0,8150.0,29287.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
134,15545.0,4232.0,55.0,0.0,593.0,4.0,0.4,16105.0,34283.0,29242.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
174,10338.0,3810.0,26.0,3.0,720.0,2.0,0.3,10546.0,46110.0,2049.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
219,4869.0,3564.0,57.0,3.0,719.0,1.0,0.2,4368.0,45213.0,34522.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
342,18943.0,686.0,38.0,3.0,773.0,0.0,0.1,358.0,36366.0,37776.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


### Feature Scaling 

In [6]:
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
X_test_scaled

array([[-1.00959496,  1.00722575, -0.80732927, ..., -0.40347329,
        -0.85972695, -0.31207962],
       [ 0.95709732, -0.33861922,  1.40832567, ..., -0.40347329,
        -0.85972695,  3.20431048],
       [-0.10355175, -0.4877257 , -1.26892405, ..., -0.40347329,
         1.16316   , -0.31207962],
       ...,
       [ 0.84404561,  0.07937358, -0.34573449, ..., -0.40347329,
        -0.85972695,  3.20431048],
       [ 1.0968333 , -1.36893792,  0.02354133, ..., -0.40347329,
         1.16316   , -0.31207962],
       [-0.91385748,  1.26939875,  1.22368776, ..., -0.40347329,
        -0.85972695, -0.31207962]], shape=(200, 27))

## Without Feature Engineering

In [8]:
# Logisticregression 
log_model = LogisticRegression(class_weight='balanced')
log_model.fit(X_train_scaled, y_train)

log_y_pred = log_model.predict(X_test_scaled) 

probs = log_model.predict_proba(X_test_scaled)[:, 1]
log_y_pred_tuned = (probs >= 0.4).astype(int)  

print(f"First 5 predictions: {log_y_pred[:5]}")
print(f"First 5 probabilities: {probs[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test, log_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test, log_y_pred)}")
print(f"Recall Score: {recall_score(y_test, log_y_pred)}")
print(f"F1 Score: {f1_score(y_test, log_y_pred)}")
print(f"CM: {confusion_matrix(y_test, log_y_pred)}")
print(f"Classification Report: {classification_report(y_test, log_y_pred)}")


First 5 predictions: [0 0 1 1 1]
First 5 probabilities: [0.30629358 0.02171352 0.60172612 0.7225918  0.99570251]
Precision Score: 0.5842696629213483
Accuracy Score: 0.775
Recall Score: 0.8666666666666667
F1 Score: 0.697986577181208
CM: [[103  37]
 [  8  52]]
Classification Report:               precision    recall  f1-score   support

           0       0.93      0.74      0.82       140
           1       0.58      0.87      0.70        60

    accuracy                           0.78       200
   macro avg       0.76      0.80      0.76       200
weighted avg       0.82      0.78      0.78       200



In [9]:
# KNNClassifier 

param_gird = {
    'n_neighbors' : range(1,21)
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_gird, 
    cv=5, 
    scoring='f1'
)

grid_knn.fit(X_train_scaled, y_train)

print(f"Best K: {grid_knn.best_params_}")
print(f"Best Score: {grid_knn.best_score_}")

best_knn = grid_knn.best_estimator_
knn_y_pred = best_knn.predict(X_test_scaled)

print(f"First 5 predictions: {knn_y_pred[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test, knn_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test, knn_y_pred)}")
print(f"Recall Score: {recall_score(y_test, knn_y_pred)}")
print(f"F1 Score: {f1_score(y_test, knn_y_pred)}")
print(f"CM: {confusion_matrix(y_test, knn_y_pred)}")
print(f"Classification Report: {classification_report(y_test, knn_y_pred)}")


Best K: {'n_neighbors': 3}
Best Score: 0.5158289269225773
First 5 predictions: [0 0 0 0 1]
Precision Score: 0.5507246376811594
Accuracy Score: 0.735
Recall Score: 0.6333333333333333
F1 Score: 0.5891472868217055
CM: [[109  31]
 [ 22  38]]
Classification Report:               precision    recall  f1-score   support

           0       0.83      0.78      0.80       140
           1       0.55      0.63      0.59        60

    accuracy                           0.73       200
   macro avg       0.69      0.71      0.70       200
weighted avg       0.75      0.73      0.74       200



In [10]:
continuous_features = ['Applicant_Income', 'Coapplicant_Income', 'Age', 
                        'Credit_Score', 'DTI_Ratio', 'Savings', 
                        'Collateral_Value', 'Loan_Amount', 'Loan_Term']

X_nb = df[continuous_features]
X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_nb, y, test_size=0.2, random_state=42, stratify=y
)

scaler_nb = StandardScaler()
X_train_nb_scaled = scaler_nb.fit_transform(X_train_nb)
X_test_nb_scaled = scaler_nb.transform(X_test_nb)

naive_model = GaussianNB()
naive_model.fit(X_train_nb_scaled, y_train_nb)

naive_y_pred = naive_model.predict(X_test_nb_scaled)

print(f"First 5 predictions: {naive_y_pred[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test_nb, naive_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test_nb, naive_y_pred)}")
print(f"Recall Score: {recall_score(y_test_nb, naive_y_pred)}")
print(f"F1 Score: {f1_score(y_test_nb, naive_y_pred)}")
print(f"CM: {confusion_matrix(y_test_nb, naive_y_pred)}")
print(f"Classification Report: {classification_report(y_test_nb, naive_y_pred)}")


First 5 predictions: [0 0 1 1 1]
Precision Score: 0.7049180327868853
Accuracy Score: 0.825
Recall Score: 0.7166666666666667
F1 Score: 0.7107438016528925
CM: [[122  18]
 [ 17  43]]
Classification Report:               precision    recall  f1-score   support

           0       0.88      0.87      0.87       140
           1       0.70      0.72      0.71        60

    accuracy                           0.82       200
   macro avg       0.79      0.79      0.79       200
weighted avg       0.83      0.82      0.83       200



### Feature Engineering 

In [11]:
df['Total_Income'] = df['Applicant_Income'] + df['Coapplicant_Income']
df['Loan_to_Value_Ratio'] = df['Loan_Amount'] / df['Collateral_Value']
df['High_DTI_Flag'] = np.where(df['DTI_Ratio'] > 0.4, 1, 0)
df['EMI_Estimate'] = df['Loan_Amount'] / df['Loan_Term']
df['EMI_to_Income_Ratio'] = df['EMI_Estimate'] / df['Total_Income']


In [12]:
X = df.drop(columns=["Loan_Approved"], axis=1)
y = df['Loan_Approved']

In [13]:
X.head()

,Applicant_Income,Coapplicant_Income,Age,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,...,Gender_Male,Employer_Category_Government,Employer_Category_MNC,Employer_Category_Private,Employer_Category_Unemployed,Total_Income,Loan_to_Value_Ratio,High_DTI_Flag,EMI_Estimate,EMI_to_Income_Ratio
0,17795.0,1387.0,51.0,0.0,637.0,4.0,0.5,19403.0,45638.0,16619.0,...,0.0,0.0,0.0,1.0,0.0,19182.0,0.364148,1,197.845238,0.010314
1,2860.0,2679.0,46.0,3.0,621.0,2.0,0.3,2580.0,49272.0,38687.0,...,1.0,0.0,0.0,1.0,0.0,5539.0,0.785172,0,805.979167,0.145510
2,7390.0,2106.0,25.0,2.0,674.0,4.0,0.2,13844.0,6908.0,27943.0,...,0.0,1.0,0.0,0.0,0.0,9496.0,4.045020,0,388.097222,0.040870
3,13964.0,8173.0,40.0,2.0,579.0,3.0,0.3,9553.0,10844.0,27819.0,...,0.0,1.0,0.0,0.0,0.0,22137.0,2.565382,0,463.650000,0.020945
4,13284.0,4223.0,31.0,2.0,721.0,1.0,0.3,9386.0,37629.0,12741.0,...,1.0,0.0,0.0,1.0,0.0,17507.0,0.338595,0,176.958333,0.010108


#### Train test split 

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.head()

,Applicant_Income,Coapplicant_Income,Age,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,...,Gender_Male,Employer_Category_Government,Employer_Category_MNC,Employer_Category_Private,Employer_Category_Unemployed,Total_Income,Loan_to_Value_Ratio,High_DTI_Flag,EMI_Estimate,EMI_to_Income_Ratio
789,10007.0,8853.0,41.0,1.0,694.0,1.0,0.2,9940.5,43858.0,27119.0,...,1.0,1.0,0.0,0.0,0.0,18860.0,0.618336,0,564.979167,0.029956
400,10808.0,6059.0,48.0,3.0,620.0,2.0,0.1,15445.0,43075.0,20522.8,...,1.0,0.0,0.0,1.0,0.0,16867.0,0.476443,0,427.558333,0.025349
433,2698.0,778.0,43.0,3.0,782.0,4.0,0.2,14805.0,13410.0,26298.0,...,0.0,0.0,0.0,0.0,0.0,3476.0,1.961074,0,313.071429,0.090067
154,6931.0,2667.0,24.0,2.0,626.0,4.0,0.4,16799.0,16116.0,7553.0,...,1.0,0.0,0.0,0.0,0.0,9598.0,0.468665,0,314.708333,0.032789
505,13003.0,4644.0,25.0,0.0,563.0,4.0,0.6,13418.0,36724.0,26717.0,...,0.0,0.0,0.0,1.0,0.0,17647.0,0.727508,1,318.059524,0.018023


In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled

array([[-0.17097538,  1.29413205,  0.11586029, ..., -0.61781273,
        -0.08853231, -0.28255138],
       [-0.00781427,  0.30691996,  0.76209298, ..., -0.61781273,
        -0.3060007 , -0.36717584],
       [-1.65979504, -1.55903102,  0.3004982 , ..., -0.61781273,
        -0.48717616,  0.82143056],
       ...,
       [ 1.33943615,  0.8574126 ,  0.76209298, ...,  1.61861345,
        -0.82916559, -0.76160907],
       [ 0.71205011, -0.29727928,  1.03904985, ...,  1.61861345,
         1.30812235,  0.58963955],
       [-0.30480415,  0.38995319,  0.20817925, ..., -0.61781273,
        -0.49265211, -0.46924968]], shape=(800, 32))

In [16]:
# Logisticregression 
log_model = LogisticRegression(class_weight='balanced')
log_model.fit(X_train_scaled, y_train)

log_y_pred = log_model.predict(X_test_scaled) 

probs = log_model.predict_proba(X_test_scaled)[:, 1]
log_y_pred_tuned = (probs >= 0.4).astype(int)  

print(f"First 5 predictions: {log_y_pred[:5]}")
print(f"First 5 probabilities: {probs[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test, log_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test, log_y_pred)}")
print(f"Recall Score: {recall_score(y_test, log_y_pred)}")
print(f"F1 Score: {f1_score(y_test, log_y_pred)}")
print(f"CM: {confusion_matrix(y_test, log_y_pred)}")
print(f"Classification Report: {classification_report(y_test, log_y_pred)}")


First 5 predictions: [0 0 1 1 1]
First 5 probabilities: [0.10608718 0.02894232 0.67664554 0.6780714  0.99729547]
Precision Score: 0.6511627906976745
Accuracy Score: 0.83
Recall Score: 0.9333333333333333
F1 Score: 0.7671232876712328
CM: [[110  30]
 [  4  56]]
Classification Report:               precision    recall  f1-score   support

           0       0.96      0.79      0.87       140
           1       0.65      0.93      0.77        60

    accuracy                           0.83       200
   macro avg       0.81      0.86      0.82       200
weighted avg       0.87      0.83      0.84       200



In [17]:
# KNNClassifier 

param_gird = {
    'n_neighbors' : range(1,21)
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_gird, 
    cv=5, 
    scoring='f1'
)

grid_knn.fit(X_train_scaled, y_train)

print(f"Best K: {grid_knn.best_params_}")
print(f"Best Score: {grid_knn.best_score_}")

best_knn = grid_knn.best_estimator_
knn_y_pred = best_knn.predict(X_test_scaled)

print(f"First 5 predictions: {knn_y_pred[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test, knn_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test, knn_y_pred)}")
print(f"Recall Score: {recall_score(y_test, knn_y_pred)}")
print(f"F1 Score: {f1_score(y_test, knn_y_pred)}")
print(f"CM: {confusion_matrix(y_test, knn_y_pred)}")
print(f"Classification Report: {classification_report(y_test, knn_y_pred)}")


Best K: {'n_neighbors': 11}
Best Score: 0.6022500492200578
First 5 predictions: [0 0 1 1 1]
Precision Score: 0.6290322580645161
Accuracy Score: 0.78
Recall Score: 0.65
F1 Score: 0.639344262295082
CM: [[117  23]
 [ 21  39]]
Classification Report:               precision    recall  f1-score   support

           0       0.85      0.84      0.84       140
           1       0.63      0.65      0.64        60

    accuracy                           0.78       200
   macro avg       0.74      0.74      0.74       200
weighted avg       0.78      0.78      0.78       200



In [18]:
# Naive Bayes



continuous_features = ['Applicant_Income', 'Coapplicant_Income', 'Total_Income', 'Age', 
                        'Credit_Score', 'DTI_Ratio', 'Savings', 
                        'Collateral_Value', 'Loan_Amount', 'Loan_Term']

X_nb = df[continuous_features]
X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_nb, y, test_size=0.2, random_state=42, stratify=y
)

scaler_nb = StandardScaler()
X_train_nb_scaled = scaler_nb.fit_transform(X_train_nb)
X_test_nb_scaled = scaler_nb.transform(X_test_nb)

naive_model = GaussianNB()
naive_model.fit(X_train_nb_scaled, y_train_nb)

naive_y_pred = naive_model.predict(X_test_nb_scaled)


print(f"First 5 predictions: {naive_y_pred[:5]}")

# Evaluation metrics 
print(f"Precision Score: {precision_score(y_test_nb, naive_y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test_nb, naive_y_pred)}")
print(f"Recall Score: {recall_score(y_test_nb, naive_y_pred)}")
print(f"F1 Score: {f1_score(y_test_nb, naive_y_pred)}")
print(f"CM: {confusion_matrix(y_test_nb, naive_y_pred)}")
print(f"Classification Report: {classification_report(y_test_nb, naive_y_pred)}")


First 5 predictions: [0 0 1 0 1]
Precision Score: 0.7413793103448276
Accuracy Score: 0.84
Recall Score: 0.7166666666666667
F1 Score: 0.7288135593220338
CM: [[125  15]
 [ 17  43]]
Classification Report:               precision    recall  f1-score   support

           0       0.88      0.89      0.89       140
           1       0.74      0.72      0.73        60

    accuracy                           0.84       200
   macro avg       0.81      0.80      0.81       200
weighted avg       0.84      0.84      0.84       200

